# VieNeuTTS Kaggle T4x2 v29 - private repo + native infer segments

This notebook clones the private repo branch on Kaggle by reading `GITHUB_USERNAME` and `GITHUB_TOKEN` from Kaggle User Secrets.

Main mode here is `segments_native_infer`, which uses `FastVieNeuTTS.infer_segments()` for long-term voice consistency on JSON segments while keeping per-segment SRT.


In [ ]:
# ======================================================
# 1. KAGGLE PATH CONFIG
# ======================================================
from pathlib import Path
import os, json, shutil, zipfile, subprocess, time, sys, math, re, stat

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
REPO_DIR = KAGGLE_WORKING / "VieNeu-TTS"
REPO_GIT_URL = "https://github.com/darkdaragon/VieNeu-TTS.git"
REPO_GIT_BRANCH = "feature/my-feature"
GITHUB_SECRET_USERNAME_KEY = "GITHUB_USERNAME"
GITHUB_SECRET_TOKEN_KEY = "GITHUB_TOKEN"
APP_ROOT = KAGGLE_WORKING / "vieneutts_app"
PACKAGE_NAME = "vieneutts_pure_colab_no_gradio_package"
PACKAGE_DIR = APP_ROOT / PACKAGE_NAME
OUTPUT_ROOT = KAGGLE_WORKING / "VieNeuTTS_Output_Kaggle"
HF_CACHE_ROOT = KAGGLE_WORKING / "hf_cache" / "vieneutts_pure"
CONFIG_DIR = KAGGLE_WORKING / "vieneutts_configs"
LOG_DIR = KAGGLE_WORKING / "vieneutts_logs"
PACKAGE_REPO_ROOT = REPO_DIR / "COLAB_KAGGLE"
PACKAGE_REPO_DIR = PACKAGE_REPO_ROOT / PACKAGE_NAME

for p in [OUTPUT_ROOT, HF_CACHE_ROOT, APP_ROOT, CONFIG_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("KAGGLE_INPUT   :", KAGGLE_INPUT)
print("KAGGLE_WORKING :", KAGGLE_WORKING)
print("PACKAGE_REPO   :", PACKAGE_REPO_DIR)
print("GitHub secret username key:", GITHUB_SECRET_USERNAME_KEY)
print("GitHub secret token key   :", GITHUB_SECRET_TOKEN_KEY)


In [ ]:
# ======================================================
# 2. DEFINE PACKAGE PATH - USE REPO FOLDER COLAB_KAGGLE
# ======================================================
print("=" * 80)
print("Package path will be resolved from the cloned private repo")
print("=" * 80)

print("Repo package root:", PACKAGE_REPO_ROOT)
print("Repo package dir :", PACKAGE_REPO_DIR)
APP_PY = PACKAGE_DIR / "app.py"
RUNNER_PATH = PACKAGE_DIR / "pure_runner.py"
print("Workspace package dir:", PACKAGE_DIR)
print("Runner path         :", RUNNER_PATH)
print("Notebook will clone the private repo by using Kaggle User Secrets + .netrc when needed.")


In [ ]:
# ======================================================
# 3. CLONE / PREPARE VIENEU-TTS GPU ENV
# ======================================================
# Theo repo pnnbao97/VieNeu-TTS hiện tại:
#   Minimal / Turbo CPU:
#       uv sync
#
#   Full GPU / Standard / Fast:
#       uv sync --group gpu
#
# Cell này bám đúng flow đó.
#
# Fix gộp cho Kaggle:
#   - Đọc private GitHub credentials từ Kaggle User Secrets
#   - Ghi ~/.netrc để git clone private repo ổn định hơn
#   - Xóa env cũ /kaggle/working/python_deps khỏi PYTHONPATH
#   - Xóa folder python_deps cũ nếu có, tránh import nhầm lmdeploy cũ
#   - uv sync --group gpu
#   - Fix libcusparseLt.so.0 nếu thiếu
#   - Fix ncclCommWindowRegister bằng cách ưu tiên CUDA libs trong .venv
#   - Cài wrapt vào .venv để tránh warning Kaggle sitecustomize
#
# Không dùng:
#   - pip --target python_deps
#   - PYTHONPATH trỏ tới python_deps
#   - cài lmdeploy lẻ ngoài uv sync --group gpu

import os
import sys
import stat
import shutil
import subprocess
from pathlib import Path

if "REPO_DIR" not in globals():
    REPO_DIR = Path("/kaggle/working/VieNeu-TTS")

REPO_DIR = Path(REPO_DIR)
VENV_DIR = REPO_DIR / ".venv"
VENV_PYTHON = VENV_DIR / "bin" / "python"
VENV_SITE_PACKAGES = VENV_DIR / "lib" / "python3.12" / "site-packages"
OLD_PY_DEPS = Path("/kaggle/working/python_deps")

def clean_old_pythonpath_pollution():
    print("=" * 80)
    print("Cleaning old python_deps / PYTHONPATH pollution")
    print("=" * 80)

    old_pythonpath = os.environ.get("PYTHONPATH", "")
    print("OLD PYTHONPATH:", old_pythonpath)

    clean_parts = [
        p for p in old_pythonpath.split(":")
        if p and "/kaggle/working/python_deps" not in p
    ]

    if clean_parts:
        os.environ["PYTHONPATH"] = ":".join(clean_parts)
    else:
        os.environ.pop("PYTHONPATH", None)

    print("NEW PYTHONPATH:", os.environ.get("PYTHONPATH", ""))

    if OLD_PY_DEPS.exists():
        print("Removing old python_deps:", OLD_PY_DEPS)
        shutil.rmtree(OLD_PY_DEPS)
    else:
        print("No old python_deps folder found.")

def read_kaggle_secret(secret_name):
    value = os.environ.get(secret_name, "").strip()
    if value:
        print(f"Using env secret: {secret_name}")
        return value
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as e:
        print(f"kaggle_secrets unavailable for {secret_name}: {e}")
        return ""
    try:
        value = (UserSecretsClient().get_secret(secret_name) or "").strip()
        if value:
            print(f"Loaded Kaggle User Secret: {secret_name}")
        else:
            print(f"Kaggle User Secret empty: {secret_name}")
        return value
    except Exception as e:
        print(f"Kaggle User Secret not available: {secret_name} ({e})")
        return ""

def configure_private_repo_auth():
    if REPO_DIR.exists():
        print("Repo exists, skip auth bootstrap:", REPO_DIR)
        return

    github_user = read_kaggle_secret(GITHUB_SECRET_USERNAME_KEY)
    github_token = read_kaggle_secret(GITHUB_SECRET_TOKEN_KEY)

    if not github_token:
        raise RuntimeError(
            "Private repo clone requires Kaggle User Secret GITHUB_TOKEN. "
            "Optionally add GITHUB_USERNAME too."
        )

    if not github_user:
        github_user = "x-access-token"
        print("GITHUB_USERNAME missing. Fallback login='x-access-token'.")

    netrc = Path.home() / ".netrc"
    netrc.write_text(
        "\n".join([
            "machine github.com",
            f"login {github_user}",
            f"password {github_token}",
            "machine api.github.com",
            f"login {github_user}",
            f"password {github_token}",
            "",
        ]),
        encoding="utf-8",
    )
    os.chmod(netrc, stat.S_IRUSR | stat.S_IWUSR)
    os.environ["GIT_TERMINAL_PROMPT"] = "0"
    print("Wrote ~/.netrc for private GitHub clone:", netrc)

def run_cmd(cmd, allow_fail=False, extra_env=None):
    print("=" * 80)
    print(cmd)

    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)

    if "PYTHONPATH" in env and "/kaggle/working/python_deps" in env["PYTHONPATH"]:
        env["PYTHONPATH"] = ":".join(
            p for p in env["PYTHONPATH"].split(":")
            if p and "/kaggle/working/python_deps" not in p
        )

    result = subprocess.run(
        ["bash", "-lc", f"cd {REPO_DIR} && {cmd}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

    print(result.stdout)
    print("RETURN:", result.returncode)

    if result.returncode != 0 and not allow_fail:
        raise RuntimeError("Command failed: " + cmd)

    return result

def collect_venv_nvidia_lib_dirs():
    lib_dirs = []
    search_roots = [
        VENV_SITE_PACKAGES / "nvidia",
        VENV_SITE_PACKAGES / "torch" / "lib",
    ]
    interesting_prefixes = (
        "libnccl", "libcusparseLt", "libcusparse", "libcublas",
        "libcudnn", "libcurand", "libcufft", "libnvrtc", "libnvJitLink",
    )

    for root in search_roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*.so*"):
                if p.name.startswith(interesting_prefixes):
                    lib_dirs.append(p.parent)
        except Exception as e:
            print("Skip searching:", root, e)

    result = []
    seen = set()
    for p in lib_dirs:
        sp = str(p)
        if sp not in seen:
            seen.add(sp)
            result.append(sp)
    return result

def apply_venv_ld_library_path():
    venv_lib_dirs = collect_venv_nvidia_lib_dirs()

    print("=" * 80)
    print("Collected .venv NVIDIA library dirs:")
    if venv_lib_dirs:
        for d in venv_lib_dirs:
            print(" -", d)
    else:
        print("No .venv NVIDIA library dirs found yet.")

    old_ld = os.environ.get("LD_LIBRARY_PATH", "")
    parts = venv_lib_dirs + ([old_ld] if old_ld else [])
    os.environ["LD_LIBRARY_PATH"] = ":".join(parts)

    print("=" * 80)
    print("Applied LD_LIBRARY_PATH:")
    print(os.environ["LD_LIBRARY_PATH"])

def test_gpu_imports():
    apply_venv_ld_library_path()

    test_cmd = """unset PYTHONPATH
uv run python - <<'PY'
import os
import sys

print("python:", sys.executable)
print("PYTHONPATH:", os.environ.get("PYTHONPATH", ""))
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH", ""))

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("cuda count:", torch.cuda.device_count())

import lmdeploy
print("lmdeploy file:", lmdeploy.__file__)
print("lmdeploy OK")

from vieneu import Vieneu
print("vieneu OK")
PY"""
    return run_cmd(test_cmd, allow_fail=True)

clean_old_pythonpath_pollution()
configure_private_repo_auth()

if not REPO_DIR.exists():
    print("Cloning private VieNeu-TTS branch...")
    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"
    subprocess.run(
        ["git", "clone", "--branch", REPO_GIT_BRANCH, REPO_GIT_URL, str(REPO_DIR)],
        check=True,
        env=clone_env,
    )
else:
    print("Repo exists:", REPO_DIR)

PACKAGE_REPO_ROOT = REPO_DIR / "COLAB_KAGGLE"
PACKAGE_REPO_DIR = PACKAGE_REPO_ROOT / PACKAGE_NAME
if not PACKAGE_REPO_DIR.exists():
    raise FileNotFoundError(f"Repo package dir missing: {PACKAGE_REPO_DIR}")
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
shutil.copytree(PACKAGE_REPO_DIR, PACKAGE_DIR)
print("Copied package from repo:", PACKAGE_REPO_DIR)
APP_PY = PACKAGE_DIR / "app.py"
RUNNER_PATH = PACKAGE_DIR / "pure_runner.py"
for p in [APP_PY, RUNNER_PATH]:
    r = subprocess.run([sys.executable, "-m", "py_compile", str(p)], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout); print(r.stderr)
        raise RuntimeError(f"Syntax check failed: {p}")
    print("Syntax OK:", p.name)

uv_check = subprocess.run(
    ["bash", "-lc", "command -v uv"],
    capture_output=True,
    text=True,
)

if uv_check.returncode != 0:
    print("Installing uv...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "uv"],
        check=True,
    )
else:
    print("uv:", uv_check.stdout.strip())

print("=" * 80)
print("Preparing VieNeu-TTS GPU environment with: uv sync --group gpu")
print("=" * 80)

run_cmd("git checkout -- pyproject.toml || true", allow_fail=True)
run_cmd("UV_LINK_MODE=copy uv sync --group gpu")

if not VENV_PYTHON.exists():
    raise FileNotFoundError(f"Không thấy venv python: {VENV_PYTHON}")

print("=" * 80)
print("Ensuring NVIDIA CUDA runtime libs in .venv")
print("=" * 80)

run_cmd(
    f'uv pip install --python "{VENV_PYTHON}" nvidia-nccl-cu12 nvidia-cusparselt-cu12 wrapt',
    allow_fail=False,
)

result = test_gpu_imports()

if result.returncode != 0:
    output = result.stdout or ""

    if "ncclCommWindowRegister" in output or "libnccl" in output:
        print("=" * 80)
        print("Detected NCCL mismatch. Reinstalling nvidia-nccl-cu12 into .venv...")
        print("=" * 80)
        run_cmd(
            f'uv pip install --python "{VENV_PYTHON}" --reinstall-package nvidia-nccl-cu12 nvidia-nccl-cu12',
            allow_fail=False,
        )
        result = test_gpu_imports()

if result.returncode != 0:
    output = result.stdout or ""

    if "libcusparseLt.so.0" in output or "cusparseLt" in output:
        print("=" * 80)
        print("Detected missing libcusparseLt. Reinstalling nvidia-cusparselt-cu12 into .venv...")
        print("=" * 80)
        run_cmd(
            f'uv pip install --python "{VENV_PYTHON}" --reinstall-package nvidia-cusparselt-cu12 nvidia-cusparselt-cu12',
            allow_fail=False,
        )
        result = test_gpu_imports()

if result.returncode != 0:
    output = result.stdout or ""

    if "No module named 'wrapt'" in output or "ModuleNotFoundError: No module named 'wrapt'" in output:
        print("=" * 80)
        print("Detected missing wrapt. Installing wrapt into .venv...")
        print("=" * 80)
        run_cmd(f'uv pip install --python "{VENV_PYTHON}" wrapt', allow_fail=False)
        result = test_gpu_imports()

if result.returncode != 0:
    print("=" * 80)
    print("❌ VieNeu-TTS GPU environment test failed.")
    raise RuntimeError("VieNeu-TTS GPU environment test failed. Xem log phía trên.")

print("✅ VieNeu-TTS GPU environment ready.")


In [ ]:
# ======================================================
# 4. CHECK GPUS AND RAM
# ======================================================
subprocess.run(["bash", "-lc", "nvidia-smi || true"], check=False)
try:
    import psutil
    vm = psutil.virtual_memory()
    print("System RAM total GB:", round(vm.total / 1024**3, 2))
    print("System RAM available GB:", round(vm.available / 1024**3, 2))
except Exception as e:
    print("RAM check failed:", e)
try:
    import torch
    print("torch.cuda.device_count():", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
except Exception as e:
    print("Torch GPU check failed:", e)

In [ ]:
# ======================================================
# 5. SELECT INPUT FILES — SAFE JSON/TXT ONLY
# ======================================================
# Chỉ lấy:
#   - *_audio_segments.json
#   - file .txt truyện thật
#
# Không lấy:
#   - requirements.txt
#   - README / LICENSE
#   - file nằm trong package vieneutts_pure_colab_no_gradio_package
#   - file thuộc repo VieNeu-TTS
#   - config json / notebook metadata / package files

from pathlib import Path

def is_inside_package_or_repo(path):
    s = str(path).replace("\\", "/").lower()
    blocked_parts = [
        "vieneutts_pure_colab_no_gradio_package",
        "/vieneu-tts/",
        "/vieneu-tts/",
        "site-packages",
        "__pycache__",
    ]
    return any(part in s for part in blocked_parts)

def is_story_txt(path):
    name = path.name.lower()
    s = str(path).replace("\\", "/").lower()

    blocked_names = {
        "requirements.txt",
        "readme.txt",
        "license.txt",
        "notice.txt",
        "changelog.txt",
    }

    if name in blocked_names:
        return False

    if is_inside_package_or_repo(path):
        return False

    # Loại txt quá nhỏ, thường là metadata/dependency note.
    try:
        if path.stat().st_size < 200:
            return False
    except Exception:
        return False

    return True

def is_audio_segments_json(path):
    name = path.name.lower()
    if not name.endswith("_audio_segments.json"):
        return False
    if is_inside_package_or_repo(path):
        return False
    return True

input_candidates = []

# 1) Ưu tiên JSON segments thật
input_candidates += [p for p in sorted(KAGGLE_INPUT.glob("**/*_audio_segments.json")) if is_audio_segments_json(p)]

# 2) TXT truyện thuần
input_candidates += [p for p in sorted(KAGGLE_INPUT.glob("**/*.txt")) if is_story_txt(p)]

# Deduplicate
seen = set()
SELECTED_INPUT_FILES = []
for p in input_candidates:
    sp = str(p)
    if sp not in seen:
        seen.add(sp)
        SELECTED_INPUT_FILES.append(p)

if not SELECTED_INPUT_FILES:
    raise RuntimeError("Không tìm thấy *_audio_segments.json hoặc .txt truyện hợp lệ trong /kaggle/input.")

print("Selected input files:")
for i, p in enumerate(SELECTED_INPUT_FILES, start=1):
    print(f"{i:02d}. {p}")

print()
print("Ignored package/dependency files are intentionally not listed.")

In [ ]:
# ======================================================
# 6. CONFIG RENDER
# ======================================================
BACKEND_MODE = "fast"          # "fast" hoặc "standard"
QUALITY_MODE = "full_quality_v2"
DEVICE_MODE = "gpu"
ENGINE_STRATEGY = "shared_single_engine"
RENDER_ORDER_STRATEGY = "non_dialogue_then_dialogue"
WORKER_COUNT_PER_GPU = 1
MAX_GPUS_TO_USE = 2
MEMORY_UTIL = 0.7
TTS_TIMEOUT_SEC = 450
TIMELINE_MODE = "fast"
SUBTITLE_FORMAT = "srt"

# Final audio speed. 1.00 = giữ nguyên, 1.08/1.12 = nhanh hơn nhẹ.
AUDIO_SPEED = 1.08


# Render input mode:
#   "segments"                         = old behavior, render every segment separately
#   "segments_ref_chunks"              = input/SRT vẫn theo segments; gom non-dialogue liền kề vào 1 infer, rồi cắt audio về từng segment
#   "segments_single_voice_ref_chunks" = input/SRT vẫn theo segments; ép 1 giọng như TXT, infer chung nhiều segment, rồi cắt audio về từng segment
#   "segments_native_infer"            = input/SRT theo segments; gọi FastVieNeuTTS.infer_segments() native để route narr/doan, nữ/doan, nam/vinh
#   "tag_long_text"                    = JSON: gom thành long text block; SRT theo block, không còn chi tiết từng segment gốc
#   "txt_long_text"                    = TXT: one single-voice long text; SDK handles split/join
RENDER_INPUT_MODE = "segments_native_infer"
TAG_KEEP_DIALOGUE_SEPARATE = "1"
TAG_NON_DIALOGUE_VOICE = "doan"
TAG_DIALOGUE_FEMALE_VOICE = "doan"
TAG_DIALOGUE_MALE_VOICE = "vinh"
TAG_NON_DIALOGUE_EMOTION = "storytelling"
TAG_DIALOGUE_EMOTION = "natural"

# segments_ref_chunks mode:
# - Gom TẤT CẢ segment non-dialogue liền kề, không giới hạn số segment.
# - Một block non-dialogue dùng chung reference voice/emotion để giữ tone giống long text.
# - Dialogue vẫn tách riêng để không lẫn speaker nam/nữ.
SEGMENTS_REF_CHUNKS_NON_DIALOGUE_VOICE = "doan"
SEGMENTS_REF_CHUNKS_NON_DIALOGUE_EMOTION = "storytelling"
SEGMENTS_REF_CHUNKS_NON_DIALOGUE_PAUSE_AFTER_MS = 180
SEGMENTS_REF_CHUNKS_DIALOGUE_EMOTION = "natural"

# segments_single_voice_ref_chunks mode:
# - Giống TXT một giọng, nhưng vẫn giữ từng JSON segment để build SRT.
# - Render có thể infer chung nhiều segment, sau đó cắt audio real về từng segment.
SEGMENTS_SINGLE_VOICE = "doan"                 # doan / vinh
SEGMENTS_SINGLE_VOICE_EMOTION = "storytelling" # storytelling / natural
SEGMENTS_SINGLE_VOICE_PAUSE_AFTER_MS = 180
NATIVE_SEGMENTS_NARRATOR_STABILITY_MODE = "locked_safe"
NATIVE_SEGMENTS_DIALOGUE_STABILITY_MODE = "stable"
NATIVE_SEGMENTS_GROUP_MAX_CHARS = 1200
NATIVE_SEGMENTS_BATCH_CHARS = 12000

PLAIN_TEXT_USE_AUTHOR_SPLIT = "1"

# TXT input mode: single voice only, không phân biệt thoại.
PLAIN_TEXT_VOICE = "doan"              # doan / vinh
PLAIN_TEXT_EMOTION = "storytelling"    # storytelling / natural
PLAIN_TEXT_MAX_CHARS = 420             # 350-500 ổn; cao hơn thì ít segment hơn nhưng dễ lỗi hơn
PLAIN_TEXT_MIN_CHARS = 180
PLAIN_TEXT_PAUSE_AFTER_MS = 180
PLAIN_TEXT_RATE_PCT = 0
PLAIN_TEXT_PITCH_HZ = 0

VIENEU_NARRATION_EMOTION = "storytelling"
VIENEU_DIALOGUE_EMOTION = "natural"
FORCE_RERENDER = False
RESCUE_REPEATED_SHORT_FAILURES = False
RETRY_FAILED_ONLY = False
CACHE_ONLY = False
RUN_NAME_PREFIX = "v25_t4x2"

# VieNeu reference voice stability / sampling.
# Áp dụng cho CẢ 3 RENDER_INPUT_MODE:
#   1) "segments"      = render từng segment
#   2) "tag_long_text" = JSON tag mode: non-dialogue gom long text, dialogue giữ tag
#   3) "txt_long_text" = TXT/plain long text
#
# BOX SELECT / Colab form:
#   locked_safe = KHUYẾN NGHỊ: ổn định mạnh nhưng vẫn bật sampling nhẹ để tránh đoạn im lặng/no speech token
#   locked_max  = khóa mạnh nhất: do_sample=0, top_k=1; có thể gây vài đoạn không có âm thanh
#   locked      = ổn định mạnh, còn sampling nhẹ
#   stable      = cân bằng ổn định/cảm xúc
#   balanced    = tự nhiên hơn, có thể nhảy tone hơn
#   creative    = gần default VieNeu, diễn cảm hơn nhưng kém đồng nhất hơn
#   custom      = tự sửa các giá trị VIENEU_INFER_* bên dưới
VIENEU_STABILITY_PRESET = "locked_safe"  #@param ["locked_safe", "locked_max", "locked", "stable", "balanced", "creative", "custom"]

VIENEU_STABILITY_PRESETS = {
    # KHUYẾN NGHỊ HIỆN TẠI:
    # Vẫn khóa giọng rất mạnh, nhưng do_sample=1 để model còn đường sinh speech token.
    # Tránh lỗi locked_max bị vài đoạn render ra im lặng/no audio.
    "locked_safe": {
        "mode": "locked_safe",
        "temperature": 0.28,
        "top_k": 8,
        "top_p": 0.70,
        "repetition_penalty": 1.06,
        "do_sample": "1",
    },
    # Khóa mạnh nhất có thể: dùng để test nhanh consistency, NHƯNG có thể gây đoạn im lặng.
    # Nếu gặp no audio/no speech token thì quay về locked_safe.
    "locked_max": {
        "mode": "locked_max",
        "temperature": 0.25,
        "top_k": 1,
        "top_p": 0.60,
        "repetition_penalty": 1.05,
        "do_sample": "0",
    },
    # Ổn định mạnh nhưng sampling rộng hơn locked_safe một chút.
    "locked": {
        "mode": "locked",
        "temperature": 0.30,
        "top_k": 12,
        "top_p": 0.72,
        "repetition_penalty": 1.08,
        "do_sample": "1",
    },
    # Khuyến nghị khi render truyện dài thật nếu locked_safe nghe quá đều.
    "stable": {
        "mode": "stable",
        "temperature": 0.55,
        "top_k": 20,
        "top_p": 0.80,
        "repetition_penalty": 1.10,
        "do_sample": "1",
    },
    "balanced": {
        "mode": "balanced",
        "temperature": 0.70,
        "top_k": 35,
        "top_p": 0.88,
        "repetition_penalty": 1.15,
        "do_sample": "1",
    },
    "creative": {
        "mode": "creative",
        "temperature": 1.00,
        "top_k": 50,
        "top_p": 0.95,
        "repetition_penalty": 1.20,
        "do_sample": "1",
    },
}

_preset = VIENEU_STABILITY_PRESETS.get(VIENEU_STABILITY_PRESET, VIENEU_STABILITY_PRESETS["locked_safe"])
VIENEU_VOICE_STABILITY_MODE = _preset["mode"]
VIENEU_INFER_TEMPERATURE = _preset["temperature"]
VIENEU_INFER_TOP_K = _preset["top_k"]
VIENEU_INFER_TOP_P = _preset["top_p"]
VIENEU_INFER_REPETITION_PENALTY = _preset["repetition_penalty"]
VIENEU_INFER_DO_SAMPLE = _preset["do_sample"]

# Nếu chọn VIENEU_STABILITY_PRESET = "custom", sửa trực tiếp các dòng dưới đây.
if VIENEU_STABILITY_PRESET == "custom":
    VIENEU_VOICE_STABILITY_MODE = "custom"
    VIENEU_INFER_TEMPERATURE = 0.28       # thấp hơn = ổn định hơn, nhưng quá thấp dễ phẳng/lỗi
    VIENEU_INFER_TOP_K = 8                # 8 = vẫn ổn định, an toàn hơn top_k=1
    VIENEU_INFER_TOP_P = 0.70             # thấp hơn = ít random hơn
    VIENEU_INFER_REPETITION_PENALTY = 1.06
    VIENEU_INFER_DO_SAMPLE = "1"          # 1 = an toàn hơn; 0 có thể gây đoạn im lặng

VIENEU_STANDARD_SPEED_MODE = "balanced"
VIENEU_STANDARD_MIN_NEW_TOKENS = 32
VIENEU_STANDARD_MAX_NEW_TOKENS = 768
VIENEU_STANDARD_MAX_CONTEXT = 2048
VIENEU_INFER_MAX_CHARS = 256
VIENEU_APPLY_WATERMARK = "1"

VIENEU_LMDEPLOY_DTYPE = "float16"
VIENEU_LMDEPLOY_SESSION_LEN = "12288"
VIENEU_LMDEPLOY_MAX_PREFILL_TOKEN_NUM = "4096"
VIENEU_LMDEPLOY_MAX_NEW_TOKENS = "1536"
VIENEU_LMDEPLOY_MIN_NEW_TOKENS = "24"

VIENEU_FAIL_FAST_ON_ENGINE_INIT = "1"
VIENEU_PRELOAD_SHARED_ENGINE = "1"

if BACKEND_MODE == "fast":
    ENGINE_STRATEGY = "shared_single_engine"
    WORKER_COUNT_PER_GPU = 1

STANDARD_BACKBONE_REPO = "pnnbao-ump/VieNeu-TTS-v2" if BACKEND_MODE == "standard" else ""
GGUF_FILENAME = ""
import torch
GPU_COUNT_TO_USE = max(1, min(MAX_GPUS_TO_USE, torch.cuda.device_count())) if DEVICE_MODE == "gpu" else 1
print(json.dumps({
    "backend": BACKEND_MODE,
    "worker_per_gpu": WORKER_COUNT_PER_GPU,
    "gpus": GPU_COUNT_TO_USE,
    "render_input_mode": RENDER_INPUT_MODE,
    "voice_stability_mode": VIENEU_VOICE_STABILITY_MODE,
    "infer_temperature": VIENEU_INFER_TEMPERATURE,
    "infer_top_k": VIENEU_INFER_TOP_K,
    "infer_top_p": VIENEU_INFER_TOP_P,
    "audio_speed": AUDIO_SPEED,
    "txt_voice": PLAIN_TEXT_VOICE,
    "txt_max_chars": PLAIN_TEXT_MAX_CHARS,
}, ensure_ascii=False, indent=2))

In [ ]:
# ======================================================
# 7. SPLIT FILES ACROSS GPUS
# ======================================================
def split_round_robin(items, n):
    buckets = [[] for _ in range(n)]
    for idx, item in enumerate(items):
        buckets[idx % n].append(item)
    return buckets
gpu_file_groups = split_round_robin(SELECTED_INPUT_FILES, GPU_COUNT_TO_USE)
for gpu_idx, group in enumerate(gpu_file_groups):
    print(f"GPU {gpu_idx}: {len(group)} file(s)")
    for p in group:
        print("  -", p)

In [ ]:
# ======================================================
# 8. WRITE PER-GPU CONFIGS
# ======================================================
base_run_name = time.strftime(f"{RUN_NAME_PREFIX}_%Y%m%d_%H%M%S")
per_gpu_jobs = []
for gpu_idx, file_group in enumerate(gpu_file_groups):
    if not file_group:
        continue
    run_name = f"{base_run_name}_gpu{gpu_idx}"
    cfg_path = CONFIG_DIR / f"{run_name}.json"
    log_path = LOG_DIR / f"{run_name}.log"
    hf_cache = HF_CACHE_ROOT / f"gpu{gpu_idx}"
    hf_cache.mkdir(parents=True, exist_ok=True)
    render_cfg = {
        "app_dir": str(PACKAGE_DIR),
        "output_root": str(OUTPUT_ROOT),
        "run_name": run_name,
        "input_files": [str(p) for p in file_group],
        "backend_mode": BACKEND_MODE,
        "quality_mode": QUALITY_MODE,
        "device_mode": DEVICE_MODE,
        "engine_strategy": ENGINE_STRATEGY,
        "render_order_strategy": RENDER_ORDER_STRATEGY,
        "worker_count": WORKER_COUNT_PER_GPU,
        "timeline_mode": TIMELINE_MODE,
        "subtitle_format": SUBTITLE_FORMAT,
        "tts_timeout_sec": TTS_TIMEOUT_SEC,
        "force_rerender": FORCE_RERENDER,
        "rescue_repeated_short_failures": RESCUE_REPEATED_SHORT_FAILURES,
        "retry_failed_only": RETRY_FAILED_ONLY,
        "cache_only": CACHE_ONLY,
        "audio_speed": AUDIO_SPEED,
        "plain_text_voice": PLAIN_TEXT_VOICE,
        "plain_text_emotion": PLAIN_TEXT_EMOTION,
        "plain_text_max_chars": PLAIN_TEXT_MAX_CHARS,
        "plain_text_min_chars": PLAIN_TEXT_MIN_CHARS,
        "plain_text_pause_after_ms": PLAIN_TEXT_PAUSE_AFTER_MS,
        "plain_text_rate_pct": PLAIN_TEXT_RATE_PCT,
        "plain_text_pitch_hz": PLAIN_TEXT_PITCH_HZ,

        "render_input_mode": RENDER_INPUT_MODE,
        "tag_keep_dialogue_separate": TAG_KEEP_DIALOGUE_SEPARATE,
        "tag_non_dialogue_voice": TAG_NON_DIALOGUE_VOICE,
        "tag_dialogue_female_voice": TAG_DIALOGUE_FEMALE_VOICE,
        "tag_dialogue_male_voice": TAG_DIALOGUE_MALE_VOICE,
        "tag_non_dialogue_emotion": TAG_NON_DIALOGUE_EMOTION,
        "tag_dialogue_emotion": TAG_DIALOGUE_EMOTION,
        "segments_ref_chunks_non_dialogue_voice": SEGMENTS_REF_CHUNKS_NON_DIALOGUE_VOICE,
        "segments_ref_chunks_non_dialogue_emotion": SEGMENTS_REF_CHUNKS_NON_DIALOGUE_EMOTION,
        "segments_ref_chunks_non_dialogue_pause_after_ms": SEGMENTS_REF_CHUNKS_NON_DIALOGUE_PAUSE_AFTER_MS,
        "segments_ref_chunks_dialogue_emotion": SEGMENTS_REF_CHUNKS_DIALOGUE_EMOTION,
        "segments_single_voice": SEGMENTS_SINGLE_VOICE,
        "segments_single_voice_emotion": SEGMENTS_SINGLE_VOICE_EMOTION,
        "segments_single_voice_pause_after_ms": SEGMENTS_SINGLE_VOICE_PAUSE_AFTER_MS,
        "native_segments_narrator_stability_mode": NATIVE_SEGMENTS_NARRATOR_STABILITY_MODE,
        "native_segments_dialogue_stability_mode": NATIVE_SEGMENTS_DIALOGUE_STABILITY_MODE,
        "native_segments_group_max_chars": NATIVE_SEGMENTS_GROUP_MAX_CHARS,
        "native_segments_batch_chars": NATIVE_SEGMENTS_BATCH_CHARS,
        "vieneu_voice_stability_mode": VIENEU_VOICE_STABILITY_MODE,
        "vieneu_infer_temperature": VIENEU_INFER_TEMPERATURE,
        "vieneu_infer_top_k": VIENEU_INFER_TOP_K,
        "vieneu_infer_top_p": VIENEU_INFER_TOP_P,
        "vieneu_infer_repetition_penalty": VIENEU_INFER_REPETITION_PENALTY,
        "vieneu_infer_do_sample": VIENEU_INFER_DO_SAMPLE,
        "plain_text_use_author_split": PLAIN_TEXT_USE_AUTHOR_SPLIT,
        "env": {
            "GOOGLE_DRIVE_OUTPUT_ROOT": str(OUTPUT_ROOT),
            "USE_GOOGLE_DRIVE_WHEN_NO_HF": "0",
            "HF_HOME": str(hf_cache),
            "HUGGINGFACE_HUB_CACHE": str(hf_cache),
            "HF_HUB_ENABLE_HF_TRANSFER": "1",
            "CUDA_VISIBLE_DEVICES": str(gpu_idx) if DEVICE_MODE == "gpu" else "",
            "VIENEU_BACKEND_MODE": BACKEND_MODE,
            "VIENEU_QUALITY_MODE": QUALITY_MODE,
            "VIENEU_DEVICE_MODE": DEVICE_MODE,
            "VIENEU_ENGINE_STRATEGY": ENGINE_STRATEGY,
            "VIENEU_RENDER_ORDER_STRATEGY": RENDER_ORDER_STRATEGY,
            "RENDER_INPUT_MODE": RENDER_INPUT_MODE,
            "VIENEU_RENDER_INPUT_MODE": RENDER_INPUT_MODE,
            "VIENEU_MEMORY_UTIL": str(MEMORY_UTIL),
            "VIENEU_TP": "1",
            "VIENEU_STANDARD_BACKBONE_REPO": STANDARD_BACKBONE_REPO,
            "VIENEU_GGUF_FILENAME": GGUF_FILENAME,
            "VIENEU_BACKBONE_DEVICE": "cuda" if DEVICE_MODE == "gpu" else "cpu",
            "VIENEU_CODEC_DEVICE": "cuda" if DEVICE_MODE == "gpu" else "cpu",
            "VIENEU_ALLOW_GGUF_FALLBACK": "0",
            "VIENEU_DUAL_FALLBACK_TO_SINGLE_SWITCH": "1",
            "VIENEU_EMOTION": PLAIN_TEXT_EMOTION,
            "VIENEU_INFER_SEGMENTS_NARRATOR_STABILITY_MODE": NATIVE_SEGMENTS_NARRATOR_STABILITY_MODE,
            "VIENEU_INFER_SEGMENTS_DIALOGUE_STABILITY_MODE": NATIVE_SEGMENTS_DIALOGUE_STABILITY_MODE,
            "VIENEU_INFER_SEGMENTS_GROUP_MAX_CHARS": str(NATIVE_SEGMENTS_GROUP_MAX_CHARS),
            "VIENEU_INFER_SEGMENTS_BATCH_CHARS": str(NATIVE_SEGMENTS_BATCH_CHARS),
            "VIENEU_NARRATION_EMOTION": VIENEU_NARRATION_EMOTION,
            "VIENEU_DIALOGUE_EMOTION": VIENEU_DIALOGUE_EMOTION,

                "VIENEU_NON_DIALOGUE_VOICE": "doan",
                "VIENEU_DIALOGUE_FEMALE_VOICE": "doan",
                "VIENEU_DIALOGUE_MALE_VOICE": "vinh",
            "VIENEU_VOICE_STABILITY_MODE": VIENEU_VOICE_STABILITY_MODE,
            "VIENEU_INFER_TEMPERATURE": str(VIENEU_INFER_TEMPERATURE),
            "VIENEU_INFER_TOP_K": str(VIENEU_INFER_TOP_K),
            "VIENEU_INFER_TOP_P": str(VIENEU_INFER_TOP_P),
            "VIENEU_INFER_REPETITION_PENALTY": str(VIENEU_INFER_REPETITION_PENALTY),
            "VIENEU_INFER_DO_SAMPLE": str(VIENEU_INFER_DO_SAMPLE),
            "VIENEU_SEGMENT_TEMPERATURE": str(VIENEU_INFER_TEMPERATURE),
            "VIENEU_SEGMENT_TOP_K": str(VIENEU_INFER_TOP_K),
            "VIENEU_SEGMENT_TOP_P": str(VIENEU_INFER_TOP_P),
            "VIENEU_TAG_TEMPERATURE": str(VIENEU_INFER_TEMPERATURE),
            "VIENEU_TAG_TOP_K": str(VIENEU_INFER_TOP_K),
            "VIENEU_TAG_TOP_P": str(VIENEU_INFER_TOP_P),
            "VIENEU_LONG_TEXT_TEMPERATURE": str(VIENEU_INFER_TEMPERATURE),
            "VIENEU_LONG_TEXT_TOP_K": str(VIENEU_INFER_TOP_K),
            "VIENEU_LONG_TEXT_TOP_P": str(VIENEU_INFER_TOP_P),
            "VIENEU_STANDARD_SPEED_MODE": VIENEU_STANDARD_SPEED_MODE,
            "VIENEU_STANDARD_MIN_NEW_TOKENS": str(VIENEU_STANDARD_MIN_NEW_TOKENS),
            "VIENEU_STANDARD_MAX_NEW_TOKENS": str(VIENEU_STANDARD_MAX_NEW_TOKENS),
            "VIENEU_STANDARD_MAX_CONTEXT": str(VIENEU_STANDARD_MAX_CONTEXT),
            "VIENEU_INFER_MAX_CHARS": str(VIENEU_INFER_MAX_CHARS),
            "VIENEU_APPLY_WATERMARK": str(VIENEU_APPLY_WATERMARK),
            "VIENEU_LMDEPLOY_DTYPE": str(VIENEU_LMDEPLOY_DTYPE),
            "VIENEU_LMDEPLOY_SESSION_LEN": str(VIENEU_LMDEPLOY_SESSION_LEN),
            "VIENEU_LMDEPLOY_MAX_PREFILL_TOKEN_NUM": str(VIENEU_LMDEPLOY_MAX_PREFILL_TOKEN_NUM),
            "VIENEU_LMDEPLOY_MAX_NEW_TOKENS": str(VIENEU_LMDEPLOY_MAX_NEW_TOKENS),
            "VIENEU_LMDEPLOY_MIN_NEW_TOKENS": str(VIENEU_LMDEPLOY_MIN_NEW_TOKENS),
            "VIENEU_FAIL_FAST_ON_ENGINE_INIT": str(VIENEU_FAIL_FAST_ON_ENGINE_INIT),
            "VIENEU_PRELOAD_SHARED_ENGINE": str(VIENEU_PRELOAD_SHARED_ENGINE),
            "AUDIO_SPEED": str(AUDIO_SPEED),
            "PLAIN_TEXT_VOICE": str(PLAIN_TEXT_VOICE),
            "PLAIN_TEXT_EMOTION": str(PLAIN_TEXT_EMOTION),
            "PLAIN_TEXT_MAX_CHARS": str(PLAIN_TEXT_MAX_CHARS),
            "PLAIN_TEXT_MIN_CHARS": str(PLAIN_TEXT_MIN_CHARS),
            "PLAIN_TEXT_PAUSE_AFTER_MS": str(PLAIN_TEXT_PAUSE_AFTER_MS),

                "RENDER_INPUT_MODE": RENDER_INPUT_MODE,
                "TAG_KEEP_DIALOGUE_SEPARATE": TAG_KEEP_DIALOGUE_SEPARATE,
                "VIENEU_NON_DIALOGUE_VOICE": TAG_NON_DIALOGUE_VOICE,
                "VIENEU_DIALOGUE_FEMALE_VOICE": TAG_DIALOGUE_FEMALE_VOICE,
                "VIENEU_DIALOGUE_MALE_VOICE": TAG_DIALOGUE_MALE_VOICE,
                "TAG_NON_DIALOGUE_EMOTION": TAG_NON_DIALOGUE_EMOTION,
                "TAG_DIALOGUE_EMOTION": TAG_DIALOGUE_EMOTION,
                "PLAIN_TEXT_USE_AUTHOR_SPLIT": PLAIN_TEXT_USE_AUTHOR_SPLIT,
                "SEGMENTS_SINGLE_VOICE": SEGMENTS_SINGLE_VOICE,
                "SEGMENTS_SINGLE_VOICE_EMOTION": SEGMENTS_SINGLE_VOICE_EMOTION,
                "SEGMENTS_SINGLE_VOICE_PAUSE_AFTER_MS": str(SEGMENTS_SINGLE_VOICE_PAUSE_AFTER_MS),
                "SEGMENTS_REF_CHUNKS_NON_DIALOGUE_VOICE": SEGMENTS_REF_CHUNKS_NON_DIALOGUE_VOICE,
                "SEGMENTS_REF_CHUNKS_NON_DIALOGUE_EMOTION": SEGMENTS_REF_CHUNKS_NON_DIALOGUE_EMOTION,
                "SEGMENTS_REF_CHUNKS_NON_DIALOGUE_PAUSE_AFTER_MS": str(SEGMENTS_REF_CHUNKS_NON_DIALOGUE_PAUSE_AFTER_MS),
                "SEGMENTS_REF_CHUNKS_DIALOGUE_EMOTION": SEGMENTS_REF_CHUNKS_DIALOGUE_EMOTION,
            "VIENEU_PROGRESS_NEWLINE": "0",
            "VIENEU_PROGRESS_THROTTLE_SEC": "0.5",
            "PYTHONUNBUFFERED": "1",
        },
    }
    cfg_path.write_text(json.dumps(render_cfg, ensure_ascii=False, indent=2), encoding="utf-8")
    per_gpu_jobs.append({"gpu_idx": gpu_idx, "run_name": run_name, "config_path": cfg_path, "log_path": log_path, "file_group": file_group, "cfg": render_cfg})
for job in per_gpu_jobs:
    print("GPU", job["gpu_idx"], "run", job["run_name"], "files", len(job["file_group"]), "config", job["config_path"])

In [ ]:
# ======================================================
# 9. RUN ALL GPU JOBS IN PARALLEL
# ======================================================
processes = []
for job in per_gpu_jobs:
    env = os.environ.copy()
    env.update(job["cfg"]["env"])
    env.pop("PYTHONPATH", None)
    cmd = f"cd {REPO_DIR} && PYTHONUNBUFFERED=1 uv run python -u {RUNNER_PATH} {job['config_path']}"
    print("Launching GPU", job["gpu_idx"], "cmd:", cmd)
    log_file = open(job["log_path"], "w", encoding="utf-8")
    proc = subprocess.Popen(["bash", "-lc", cmd], stdout=log_file, stderr=subprocess.STDOUT, text=True, env=env)
    processes.append({"job": job, "proc": proc, "log_file": log_file})
print("All jobs launched. Run monitor cell below.")

In [ ]:
# ======================================================
# 10. MONITOR GPU JOBS — CLEAN PROGRESS DASHBOARD
# ======================================================
import re, time
from pathlib import Path
from IPython.display import clear_output

BAR_RE = re.compile(r"\[(?P<bar>[█░]+)\]\s+(?P<pct>[\d.]+)%.*?file\s+(?P<file>\d+/\d+).*?files ok=(?P<files_ok>\d+) fail=(?P<files_fail>\d+).*?RUNNING\s+(?P<phase>\w+).*?seg\s+(?P<seg>\d+/\d+)\s+ok=(?P<ok>\d+)\s+fail=(?P<fail>\d+).*?current=(?P<current>\S+)")
ENGINE_RE = re.compile(r"(Initializing VieNeuTTS|engine\[.*?\] init done|COMPLETE|INCOMPLETE|FAILED|ERROR|Traceback)", re.IGNORECASE)

def _read_tail_text(log_path, max_bytes=512_000):
    p = Path(log_path)
    if not p.exists():
        return ""
    size = p.stat().st_size
    with p.open("rb") as f:
        if size > max_bytes:
            f.seek(size - max_bytes)
        data = f.read()
    return data.decode("utf-8", errors="ignore")

def latest_progress(log_path):
    for line in reversed(_read_tail_text(log_path).splitlines()):
        m = BAR_RE.search(line)
        if m:
            return m.groupdict()
    return None

def latest_events(log_path, limit=4):
    hits = []
    for line in reversed(_read_tail_text(log_path).splitlines()):
        if ENGINE_RE.search(line):
            hits.append(line.strip())
        if len(hits) >= limit:
            break
    return list(reversed(hits))

while True:
    clear_output(wait=True)
    all_done = True
    print("=" * 120)
    print("KAGGLE T4x2 CLEAN PROGRESS DASHBOARD")
    print("=" * 120)
    for item in processes:
        job, proc = item["job"], item["proc"]
        ret = proc.poll()
        if ret is None:
            all_done = False
        print(f"GPU {job['gpu_idx']} | run={job['run_name']} | pid={proc.pid} | returncode={ret}")
        print("log:", job["log_path"])
        prog = latest_progress(job["log_path"])
        if prog:
            print(f"[{prog['bar']}] {prog['pct']}% | file {prog['file']} | files ok={prog['files_ok']} fail={prog['files_fail']} | phase={prog['phase']} | seg {prog['seg']} ok={prog['ok']} fail={prog['fail']} | current={prog['current']}")
        else:
            print("Waiting for progress bar...")
        ev = latest_events(job["log_path"])
        if ev:
            print("events:")
            for e in ev:
                print("  " + (e[:180] + "..." if len(e) > 180 else e))
        print("-" * 120)
    if all_done:
        print("✅ All jobs finished. Run WAIT FOR COMPLETION cell.")
        break
    time.sleep(2)

In [ ]:
# ======================================================
# 11. WAIT FOR COMPLETION
# ======================================================
results = []
for item in processes:
    ret = item["proc"].wait()
    item["log_file"].close()
    results.append({"gpu_idx": item["job"]["gpu_idx"], "run_name": item["job"]["run_name"], "returncode": ret, "log_path": str(item["job"]["log_path"])})
print(json.dumps(results, ensure_ascii=False, indent=2))
if any(r["returncode"] != 0 for r in results):
    print("❌ Some jobs failed.")
else:
    print("✅ All jobs completed.")

In [ ]:
# ======================================================
# 12. PACKAGE OUTPUTS FOR DOWNLOAD
# ======================================================
import shutil
zip_paths = []
for job in per_gpu_jobs:
    run_dir = OUTPUT_ROOT / "runs" / job["run_name"]
    zip_out = KAGGLE_WORKING / f"{job['run_name']}_outputs.zip"
    if run_dir.exists():
        if zip_out.exists():
            zip_out.unlink()
        shutil.make_archive(str(zip_out.with_suffix("")), "zip", run_dir)
        zip_paths.append(zip_out)
print("Created output zips:")
for p in zip_paths:
    print(" -", p, round(p.stat().st_size / 1024 / 1024, 2), "MB")